# Delhi AQI — Live T+1 Forecast Demo

Interactive shadow demonstration of the frozen Delhi AQI next-hour ensemble.

## Run the demo
1. Open this notebook in Google Colab.
2. Ensure your Google Drive contains the original `Delhi-AQI-Prediction` project folder and frozen model artifacts.
3. Run the single launcher cell below.
4. Select one of the ten supported stations and click **Predict Next-Hour AQI**.

The launcher loads the existing frozen Extra Trees and Random Forest models. It does not retrain or replace them.

**Data limitation:** the live source currently supplies station AQI but not the pollutant measurements required by the full 73-feature contract. Missing inputs are imputed by the existing fitted pipelines. Forecasts are labelled `SHADOW_ONLY_NOT_PRODUCTION`; the production release gate remains disabled.

**Hosting:** the Gradio sharing link is temporary and works only while the Colab runtime is active. Model binaries and datasets are intentionally excluded from this repository.


In [ ]:
# =============================================================================
# DELHI AQI — LIVE T+1 DASHBOARD
# Standalone launcher for a fresh Google Colab runtime.
#
# Prerequisite:
#   The original Delhi-AQI-Prediction project folder must exist in Google Drive,
#   including the frozen model files excluded from GitHub due to their size.
#
# Every Predict click fetches fresh live history and runs the frozen ensemble.
# Results remain SHADOW_ONLY_NOT_PRODUCTION because live pollutant values
# are unavailable from the current public source.
# =============================================================================

from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
import importlib
import importlib.util
import os
import subprocess
import sys
import json

# Install only the dashboard/runtime packages if they are missing.
missing_packages = [
    pip_name
    for module_name, pip_name in [
        ("gradio", "gradio"),
        ("fastapi", "fastapi"),
        ("psutil", "psutil"),
    ]
    if importlib.util.find_spec(module_name) is None
]

if missing_packages:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        *missing_packages
    ])

import gradio as gr
import numpy as np
import pandas as pd
import psutil
from google.colab import drive

drive.mount("/content/drive")

ROOT = Path("/content/drive/MyDrive/Delhi-AQI-Prediction")
PACKAGE = ROOT / "deployment/phase16_33_frozen_model_package"
API_DIR = ROOT / "deployment/phase17_06_inference_api"
SRC_DIR = API_DIR / "src"

CONTRACT_PATH = PACKAGE / "config/feature_contract.json"
ENSEMBLE_PATH = PACKAGE / "config/ensemble_config.json"

required_files = [
    CONTRACT_PATH,
    ENSEMBLE_PATH,
    SRC_DIR / "shadow_live.py",
    SRC_DIR / "canonical_aqi_features.py",
    SRC_DIR / "frozen_aqi_inference_missing_aware_candidate.py",
    PACKAGE / "models/phase15_10_final_tuned_extra_trees_trainval.joblib",
    PACKAGE / "models/phase15_10_final_tuned_random_forest_trainval.joblib",
]

missing_files = [str(p) for p in required_files if not p.is_file()]
if missing_files:
    raise FileNotFoundError(
        "Required project/model files are missing from Google Drive:\n"
        + "\n".join(missing_files)
        + "\nRestore the original frozen artifacts; do not retrain models "
        "as a workaround."
    )

with CONTRACT_PATH.open("r", encoding="utf-8") as f:
    CONTRACT = json.load(f)
with ENSEMBLE_PATH.open("r", encoding="utf-8") as f:
    ENSEMBLE = json.load(f)

FEATURE_ORDER = CONTRACT["feature_order"]
WEIGHTS = ENSEMBLE["weights"]

if CONTRACT.get("target") != "AQI_t_plus_1":
    raise RuntimeError("Unexpected target in frozen feature contract.")
if len(FEATURE_ORDER) != 73 or len(set(FEATURE_ORDER)) != 73:
    raise RuntimeError("Frozen feature contract must contain 73 unique columns.")

if not np.isclose(
    float(WEIGHTS["tuned_extra_trees"])
    + float(WEIGHTS["tuned_random_forest"]),
    1.0,
    atol=1e-6,
):
    raise RuntimeError("Frozen ensemble weights do not sum to 1.")

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# Ensure the existing code resolves the frozen package in Drive.
os.environ["FROZEN_PACKAGE_DIR"] = str(PACKAGE)

# Avoid accidentally reusing stale modules from a previous notebook run.
for module_name in [
    "shadow_live",
    "canonical_aqi_features",
    "frozen_aqi_inference_missing_aware_candidate",
]:
    sys.modules.pop(module_name, None)

importlib.invalidate_caches()
shadow_live = importlib.import_module("shadow_live")
inference = shadow_live.inference_candidate

if Path(inference.MODEL_ROOT).resolve() != (PACKAGE / "models").resolve():
    raise RuntimeError("Inference wrapper does not point to frozen model artifacts.")

if not callable(getattr(shadow_live, "build_live_feature_frame", None)):
    raise RuntimeError("Existing live feature builder is unavailable.")

if not callable(getattr(inference, "load_frozen_models", None)):
    raise RuntimeError("Existing frozen model loader is unavailable.")

# Original fitted models are loaded once per runtime; they are not retrained.
already_loaded = (
    getattr(inference, "_EXTRA_TREES", None) is not None
    and getattr(inference, "_RANDOM_FOREST", None) is not None
)

if not already_loaded:
    available_ram_gib = psutil.virtual_memory().available / (1024 ** 3)
    print(f"Available RAM before loading frozen models: {available_ram_gib:.2f} GiB")
    if available_ram_gib < 5.5:
        raise MemoryError(
            "Insufficient available RAM for the large frozen model artifacts. "
            "Restart Colab and use a runtime with sufficient RAM."
        )
    print("Loading frozen Extra Trees and Random Forest...")
    inference.load_frozen_models()

STATIONS = shadow_live.STATIONS
STATION_ORDER = list(shadow_live.STATION_ORDER)

if len(STATION_ORDER) != 10:
    raise RuntimeError("Expected the frozen ten-station model scope.")

def run_live_prediction(station_id):
    """Fetch new history and return one station's shadow-only t+1 forecast."""
    station_id = str(station_id).strip()

    if station_id not in STATION_ORDER:
        raise ValueError(f"Unsupported station ID: {station_id}")

    # Fresh source fetch on every call.
    X, metadata = shadow_live.build_live_feature_frame()
    X = inference.validate_feature_frame(X.copy())

    if X.shape != (10, 73) or list(X.columns) != FEATURE_ORDER:
        raise RuntimeError("Live features do not match the frozen contract.")
    if X["StationId"].astype(str).duplicated().any():
        raise RuntimeError("Duplicate station rows in live feature frame.")

    source_age_hours = float(metadata["source_age_hours"])
    if not np.isfinite(source_age_hours) or not 0 <= source_age_hours <= 1:
        raise RuntimeError(f"Live forecast origin is stale: {source_age_hours:.2f} h.")

    predictions = inference.predict_aqi(X)
    output_columns = [
        "AQI_pred_extra_trees",
        "AQI_pred_random_forest",
        "AQI_t_plus_1_pred",
    ]

    if len(predictions) != 10 or any(
        column not in predictions.columns for column in output_columns
    ):
        raise RuntimeError("Frozen ensemble output schema is invalid.")

    prediction_values = predictions[output_columns].to_numpy(dtype=float)
    if not np.isfinite(prediction_values).all():
        raise RuntimeError("Non-finite prediction returned.")

    et_weight = float(WEIGHTS["tuned_extra_trees"])
    rf_weight = float(WEIGHTS["tuned_random_forest"])
    manual_ensemble = (
        et_weight * predictions["AQI_pred_extra_trees"].to_numpy(dtype=float)
        + rf_weight * predictions["AQI_pred_random_forest"].to_numpy(dtype=float)
    )

    if not np.allclose(
        manual_ensemble,
        predictions["AQI_t_plus_1_pred"].to_numpy(dtype=float),
        rtol=1e-6,
        atol=1e-5,
    ):
        raise RuntimeError("Frozen ensemble weighting verification failed.")

    indices = np.flatnonzero(X["StationId"].astype(str).to_numpy() == station_id)
    if len(indices) != 1:
        raise RuntimeError(f"Could not uniquely locate {station_id}.")

    i = int(indices[0])
    row = X.iloc[i]
    pred = predictions.iloc[i]
    current_aqi = pd.to_numeric(pd.Series([row["AQI"]]), errors="coerce").iloc[0]

    if pd.isna(current_aqi) or not np.isfinite(float(current_aqi)):
        raise RuntimeError("Current AQI is missing or invalid.")

    missing_features = int(row.isna().sum())

    return {
        "station_id": station_id,
        "station_name": STATIONS[station_id]["project_name"],
        "current_aqi": float(current_aqi),
        "extra_trees": float(pred["AQI_pred_extra_trees"]),
        "random_forest": float(pred["AQI_pred_random_forest"]),
        "ensemble": float(pred["AQI_t_plus_1_pred"]),
        "forecast_origin": str(metadata["forecast_origin_ist"]),
        "forecast_target": str(metadata["forecast_target_ist"]),
        "source_age_hours": source_age_hours,
        "snapshot_count": int(metadata["snapshot_count"]),
        "missing_features": missing_features,
        "total_features": len(FEATURE_ORDER),
        "missing_pct": round(100 * missing_features / len(FEATURE_ORDER), 2),
        "weights": WEIGHTS,
        "status": "SHADOW_ONLY_NOT_PRODUCTION",
        "input_mode": "official_station_AQI_only",
        "source": "Official Delhi AQI portal",
        "warning": (
            "Live pollutant concentrations are unavailable. The frozen fitted "
            "imputers handle missing inputs. This is a shadow demonstration, "
            "not a production-validated forecast."
        ),
        "request_time_ist": datetime.now(
            ZoneInfo("Asia/Kolkata")
        ).isoformat(timespec="seconds"),
    }

station_labels = {
    f"{STATIONS[sid]['project_name']} ({sid})": sid
    for sid in STATION_ORDER
}

def predict_callback(station_label):
    station_id = station_labels.get(station_label)
    if station_id is None:
        return "## Select a supported station.", {"status": "INVALID_STATION"}

    try:
        result = run_live_prediction(station_id)
        delta = result["ensemble"] - result["current_aqi"]

        report = f"""
# {result['station_name']}

> **SHADOW DEMO — NOT PRODUCTION**

## Predicted AQI: {result['ensemble']:.1f}

**Forecast origin (IST):** {result['forecast_origin']}  
**Forecast target (IST):** {result['forecast_target']}

### Live reading and model results

| Metric | AQI |
|---|---:|
| Latest fetched station AQI | {result['current_aqi']:.1f} |
| Extra Trees prediction | {result['extra_trees']:.2f} |
| Random Forest prediction | {result['random_forest']:.2f} |
| **Final weighted ensemble** | **{result['ensemble']:.2f}** |
| Forecast minus current AQI | {delta:+.2f} |

### Data quality

- Source: Official Delhi AQI portal
- History snapshots fetched: {result['snapshot_count']}
- Forecast-origin age: {result['source_age_hours']:.2f} hours
- Missing inputs: {result['missing_features']} of {result['total_features']} features ({result['missing_pct']:.2f}%)

### Important limitation

The live source currently exposes AQI but not the pollutant concentration measurements required by the full 73-feature model. Missing inputs are imputed by the frozen pipeline; this forecast is not production-validated.
"""
        return report, result

    except Exception as exc:
        return (
            "## Forecast unavailable\n\n"
            f"Live fetch or prediction failed: `{type(exc).__name__}: {str(exc)[:400]}`\n\n"
            "No new forecast was produced; an older forecast is not substituted.",
            {
                "status": "REQUEST_FAILED",
                "error_type": type(exc).__name__,
                "error_message": str(exc)[:800],
                "new_forecast_produced": False,
            },
        )

with gr.Blocks(title="Delhi AQI — Live T+1 Forecast") as dashboard:
    gr.Markdown(
        """
# Delhi AQI — Live Next-Hour Forecast
### Live source → canonical 73 features → frozen ensemble → t+1 prediction

Select a supported station and click **Predict Next-Hour AQI**.
Each click requests fresh source history.

> **SHADOW ONLY — NOT PRODUCTION.** Pollutant measurements are missing from
> the live source, so this demo's forecasts rely on the fitted imputers.
        """
    )

    with gr.Row():
        with gr.Column(scale=1, min_width=250):
            station_dropdown = gr.Dropdown(
                choices=list(station_labels.keys()),
                value="Anand Vihar (DL002)",
                label="Select Delhi station",
                interactive=True,
            )
            predict_button = gr.Button(
                "Predict Next-Hour AQI",
                variant="primary",
                size="lg",
            )

        with gr.Column(scale=2, min_width=360):
            forecast_output = gr.Markdown(
                "## Ready\nSelect a station and click Predict."
            )

    with gr.Accordion("Technical response and data-quality details", open=False):
        diagnostics_output = gr.JSON(label="Latest request diagnostics")

    predict_button.click(
        fn=predict_callback,
        inputs=[station_dropdown],
        outputs=[forecast_output, diagnostics_output],
    )

dashboard.queue(default_concurrency_limit=1)

print("\nLaunching dashboard. Keep this runtime active during the demo.")
launch_result = dashboard.launch(
    share=True,
    debug=False,
    show_error=True,
    theme=gr.themes.Soft(),
)

if isinstance(launch_result, tuple) and len(launch_result) >= 3:
    print("Local URL:", launch_result[1])
    print("Public demo URL:", launch_result[2] or "See Gradio output above.")
